# Detección de fraude por voz con deepfakes

Este notebook sigue el flujo usado para clasificar audios reales y audios fake usando embeddings de wav2vec. Primero se preparan las tablas en Databricks. Luego se arman las variables del modelo. Después se balancea el entrenamiento. Finalmente se entrenan y comparan tres modelos.

**Correcciones respecto a la Entrega 1:**
- GBT ahora se entrena con `df_balanced_train` (igual que LR y RF) para garantizar comparabilidad.
- Se imprime y registra la fracción de undersampling aplicada.
- Se agrega F1-score como métrica adicional.
- Se documenta la tabla de hiperparámetros de los tres modelos.
- Se exportan los resultados a un DataFrame/CSV al final del notebook.
- La evaluación con dataset externo queda indicada como trabajo futuro (ver Paso 10).


## Paso 1. Crear la base de datos y el volumen

Aquí se crea el espacio de trabajo donde van a quedar guardados los datos y las tablas del proyecto.


In [ ]:
%sql
-- se crea la base si todavía no existe
CREATE DATABASE IF NOT EXISTS deepfake_bd;

In [ ]:
%sql
-- se crea el volumen donde están los archivos parquet
CREATE VOLUME IF NOT EXISTS deepfake_bd.data;

In [ ]:
%python
# revisamos qué archivos hay dentro del volumen
# esto ayuda a confirmar que los parquet sí están cargados
dbutils.fs.ls("/Volumes/workspace/deepfake_bd/data")

## Paso 2. Cargar los datos parquet

Se cargan los archivos de entrenamiento y prueba; estos archivos ya tienen los embeddings de audio generados con wav2vec.


In [ ]:
# cargamos una muestra del parquet de entrenamiento
df = spark.read.parquet("/Volumes/workspace/deepfake_bd/data/ASVspoof2021_train_wav2vec.parquet")
display(df.limit(5))

In [ ]:
# leemos el conjunto de entrenamiento desde el volumen
df_train = spark.read.parquet("/Volumes/workspace/deepfake_bd/data/ASVspoof2021_train_wav2vec.parquet")
df_train.write.mode("overwrite").saveAsTable("deepfake_bd.deepfake_train")

In [ ]:
# hacemos lo mismo con el conjunto de prueba
# esta tabla se usará al final para evaluar los modelos
df_test = spark.read.parquet("/Volumes/workspace/deepfake_bd/data/ASVspoof2021_test_wav2vec.parquet")
df_test.write.mode("overwrite").saveAsTable("deepfake_bd.deepfake_test")

## Paso 3. Construir la variable features

Los embeddings vienen separados en muchas columnas. Spark necesita que esas columnas estén juntas en un solo vector llamado features.


In [ ]:
from pyspark.ml.feature import VectorAssembler

# cargamos la tabla de entrenamiento que ya quedó guardada
df_train = spark.table("deepfake_bd.deepfake_train")

# nos quedamos con todas las columnas que empiezan con emb_
# esas columnas son los valores del embedding de cada audio
emb_cols = [c for c in df_train.columns if c.startswith("emb_")]

# vectorassembler junta todas las columnas numéricas en un solo vector
assembler = VectorAssembler(
    inputCols=emb_cols,
    outputCol="features"
)

# se arma el vector de variables y se deja solo lo necesario para modelar
df_train_w = assembler.transform(df_train)
df_train_w = df_train_w.select("features", "label")

display(df_train_w.limit(5))

In [ ]:
# guardamos el entrenamiento ya transformado
df_train_w.write.mode("overwrite").saveAsTable("deepfake_bd.deepfake_train_features")

In [ ]:
# aplicamos el mismo assembler al conjunto de prueba
# es importante usar la misma estructura de variables
df_test = spark.table("deepfake_bd.deepfake_test")
df_test_w = assembler.transform(df_test)
df_test_w = df_test_w.select("features", "label")

display(df_test_w.limit(5))

In [ ]:
# guardamos también la prueba transformada
df_test_w.write.mode("overwrite").saveAsTable("deepfake_bd.deepfake_test_features")

## Paso 4. Preparar etiquetas y revisar clases

La variable label viene como texto. Se transforma a 1 para fake y 0 para real. También se revisa el desbalance entre clases.


In [ ]:
# cargamos las tablas que ya tienen features y label
df_train = spark.table("deepfake_bd.deepfake_train_features")
df_test = spark.table("deepfake_bd.deepfake_test_features")

In [ ]:
from pyspark.sql.functions import when

# convertimos la etiqueta a formato numérico
# fake queda como 1 y real queda como 0
df_train = df_train.withColumn("label", when(df_train.label == "fake", 1).otherwise(0))
df_test = df_test.withColumn("label", when(df_test.label == "fake", 1).otherwise(0))

In [ ]:
# contamos cuántos audios hay por clase en entrenamiento
# esto permite ver si el dataset está balanceado o no
df_train.groupBy("label").count().display()

## Paso 5. Balancear el entrenamiento

Como hay muchas más muestras fake que reales, se aplica undersampling. La idea es bajar la clase mayoritaria para que el modelo no aprenda sesgado.


In [ ]:
# separamos las dos clases para poder balancearlas
df_train1 = df_train.filter("label = 1")
df_train0 = df_train.filter("label = 0")

In [ ]:
# calculamos qué fracción de la clase fake vamos a tomar
# se usa como referencia la cantidad de audios reales
fraction = df_train0.count() / df_train1.count()

# CORRECCIÓN: se registra la fracción para reportarla en el paper
print(f"Fracción de undersampling aplicada: {fraction:.6f}")
print(f"  Muestras reales (clase 0): {df_train0.count()}")
print(f"  Muestras fake  (clase 1): {df_train1.count()}")
print(f"  Fracción = real / fake = {fraction:.6f}")

# tomamos una muestra aleatoria de la clase fake
# la semilla ayuda a que el resultado sea reproducible
df_train1_sampled = df_train1.sample(fraction=fraction, seed=42)

# juntamos la muestra fake con todos los reales
df_balanced_train = df_train1_sampled.union(df_train0)

In [ ]:
# verificamos que el entrenamiento quedó más balanceado
df_balanced_train.groupBy("label").count().display()

## Paso 6. Hiperparámetros de los modelos

Se documentan los hiperparámetros de cada modelo antes del entrenamiento.

| Modelo | Parámetro | Valor |
|--------|-----------|-------|
| Regresión Logística | maxIter | 100 (default) |
| Regresión Logística | regParam | 0.0 (default) |
| Random Forest | numTrees | 100 |
| Random Forest | maxDepth | 10 |
| GBT | maxIter | 50 |
| GBT | maxDepth | 5 |

**Nota importante:** Los tres modelos se entrenan con `df_balanced_train` para garantizar comparabilidad en los resultados.


## Paso 7. Entrenar los modelos

Se prueban tres modelos: Regresión Logística como base, Random Forest como ensamble, y GBT como modelo más flexible.

**CORRECCIÓN respecto a Entrega 1:** GBT ahora se entrena con `df_balanced_train` (igual que LR y RF), garantizando condiciones de entrenamiento comparables.


In [ ]:
from pyspark.ml.classification import LogisticRegression

# modelo base lineal
# sirve como primera comparación porque es simple y rápido
lr = LogisticRegression(featuresCol="features", labelCol="label")

# entrenamos con el dataset balanceado
lr_model = lr.fit(df_balanced_train)

# generamos predicciones sobre el test original
lr_pred = lr_model.transform(df_test)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# random forest combina varios árboles
# esto ayuda a capturar relaciones no lineales entre los embeddings
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,   # cantidad de árboles
    maxDepth=10     # profundidad máxima de cada árbol
)

# entrenamos con el dataset balanceado y predecimos en test
rf_model = rf.fit(df_balanced_train)
rf_pred = rf_model.transform(df_test)

In [ ]:
from pyspark.ml.classification import GBTClassifier

# GBT entrena árboles de forma secuencial
# cada árbol intenta corregir errores del anterior
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=50,   # número de iteraciones del boosting
    maxDepth=5    # profundidad máxima del árbol
)

# CORRECCIÓN: se entrena con df_balanced_train (antes usaba df_train)
# esto garantiza que GBT sea comparable con LR y RF
gbt_model = gbt.fit(df_balanced_train)

gbt_pred = gbt_model.transform(df_test)

## Paso 8. Evaluar precision, recall, F1-score y AUC

Se calculan métricas para comparar los modelos. En este problema el recall es importante porque interesa detectar la mayor cantidad posible de audios fake.

**CORRECCIÓN:** Se agrega F1-score como métrica adicional.


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

evaluator_mc = MulticlassClassificationEvaluator(labelCol="label")
evaluator_bin = BinaryClassificationEvaluator(labelCol="label")

results = []

for name, pred in [("Logistic Regression", lr_pred), ("Random Forest", rf_pred), ("GBT", gbt_pred)]:
    precision = evaluator_mc.setMetricName("weightedPrecision").evaluate(pred)
    recall    = evaluator_mc.setMetricName("weightedRecall").evaluate(pred)
    f1        = evaluator_mc.setMetricName("weightedFMeasure").evaluate(pred)
    auc       = evaluator_bin.evaluate(pred)
    results.append({"Modelo": name, "Precision": round(precision, 4),
                    "Recall": round(recall, 4), "F1-score": round(f1, 4), "AUC": round(auc, 4)})
    print(f"--- {name} ---")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1-score  : {f1:.4f}")
    print(f"  AUC       : {auc:.4f}")
    print()

## Paso 9. Matrices de confusión

Las matrices muestran aciertos y errores por clase. Ayudan a ver falsos positivos y falsos negativos.


In [ ]:
# agrupamos por etiqueta real y predicción
# luego pasamos a pandas para graficar con seaborn
cm_lr  = lr_pred.groupBy("label", "prediction").count().toPandas()
cm_rf  = rf_pred.groupBy("label", "prediction").count().toPandas()
cm_gbt = gbt_pred.groupBy("label", "prediction").count().toPandas()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# función para graficar la matriz de confusión
def plot_confusion(cm, title):
    pivot = cm.pivot(index="label", columns="prediction", values="count").fillna(0)
    plt.figure()
    sns.heatmap(pivot, annot=True, fmt=".0f")
    plt.title(title)
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.show()

plot_confusion(cm_lr,  "Matriz de Confusión - Logistic Regression")
plot_confusion(cm_rf,  "Matriz de Confusión - Random Forest")
plot_confusion(cm_gbt, "Matriz de Confusión - Gradient Boosted Trees")

In [ ]:
# revisamos la distribución del test
# esto ayuda a interpretar las matrices de confusión
df_test.groupBy("label").count().display()

## Paso 10. Curva ROC comparativa

La curva ROC permite comparar la capacidad de separación de los modelos. Mientras más cerca esté la curva de la esquina superior izquierda, mejor.


In [ ]:
from pyspark.ml.functions import vector_to_array
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# tomamos la probabilidad de clase 1 para cada modelo y calculamos ROC
def get_roc(pred, label="label"):
    scores = pred.select(label, vector_to_array("probability")[1].alias("prob"))
    pdf = scores.toPandas()
    fpr, tpr, _ = roc_curve(pdf[label], pdf["prob"])
    auc_val = auc(fpr, tpr)
    return fpr, tpr, auc_val

fpr_lr,  tpr_lr,  auc_lr  = get_roc(lr_pred)
fpr_rf,  tpr_rf,  auc_rf  = get_roc(rf_pred)
fpr_gbt, tpr_gbt, auc_gbt = get_roc(gbt_pred)

# gráfico comparativo de las tres curvas
plt.figure()
plt.plot(fpr_lr,  tpr_lr,  label=f"Logistic Regression (AUC = {auc_lr:.3f})")
plt.plot(fpr_rf,  tpr_rf,  label=f"Random Forest (AUC = {auc_rf:.3f})")
plt.plot(fpr_gbt, tpr_gbt, label=f"GBT (AUC = {auc_gbt:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Comparación de Modelos")
plt.legend()
plt.show()

## Paso 11. Exportar resultados

Se exportan las métricas finales como DataFrame para facilitar la actualización de la Tabla II del paper.


In [ ]:
# exportamos los resultados de la Tabla II a CSV
import pandas as pd

df_results = pd.DataFrame(results)
print("Tabla II — Comparación de modelos:")
print(df_results.to_string(index=False))

# guardamos el CSV en el volumen de Databricks
# ajustar la ruta según el entorno del grupo
# df_results.to_csv("/Volumes/workspace/deepfake_bd/data/resultados_tabla2.csv", index=False)
# print("Resultados exportados a resultados_tabla2.csv")

## Paso 12. Trabajo futuro — Evaluación con dataset externo

**Estado:** Pendiente de implementación (Entrega 2).

Para evaluar la capacidad de generalización del modelo, se propone cargar un dataset externo (por ejemplo, ASVspoof 2019 o In-the-Wild) y aplicar el mismo `assembler` para generar el vector de features, luego evaluar con el mejor modelo seleccionado (GBT o LR según las métricas finales).

```python
# TRABAJO FUTURO — esquema de implementación (no ejecutar; dataset externo no disponible aún)
#
# df_ext = spark.read.parquet("/ruta/al/dataset_externo.parquet")
# df_ext_w = assembler.transform(df_ext).select("features", "label")
# df_ext_w = df_ext_w.withColumn("label", when(df_ext_w.label == "fake", 1).otherwise(0))
# ext_pred = gbt_model.transform(df_ext_w)
# print("AUC externo:", evaluator_bin.evaluate(ext_pred))
# print("Recall externo:", evaluator_mc.setMetricName("weightedRecall").evaluate(ext_pred))
```

Este bloque debe implementarse en la Entrega 2. Los resultados obtenidos deben reportarse en la Sección IV-F del paper.
